# Comparing influence functions with the IOM for deep learning

The goal of this notebook is to compare the influence function in deep learning with the IOM.

Download data from `"https://www.kaggle.com/code/raqhea/cats-vs-dogs-pytorch-resnet18"`.

In [0]:
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as t
import torchvision
import numpy as np
import shap
import matplotlib.pyplot as plt
from PIL import Image
from torch.utils.data import DataLoader, random_split
import os
import random

# ============================================================
# 1. Config
# ============================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

DATASET_PATH = "/cats_dogs/PetImages"
CLASS_NAMES = ["Cat", "Dog"]
NUM_CLASSES = 2
BATCH_SIZE = 32
EPOCHS = 10
LR = 0.001

# ============================================================
# 3. Transforms (exactly yours)
# ============================================================
transform = t.Compose([
    t.Resize((200, 200)),
    t.ToTensor(),
    t.ConvertImageDtype(torch.float32)
])

training_transform = t.Compose([
    t.Resize((200, 200)),
    t.RandomRotation([0, 45]),
    t.RandomHorizontalFlip(),
    t.ToTensor(),
    t.ConvertImageDtype(torch.float32)
])

# ============================================================
# 4. Dataset + Splits
# ============================================================
full_dataset = torchvision.datasets.ImageFolder(DATASET_PATH, transform=transform)
print(f"Total images: {len(full_dataset)}")
print(f"Classes: {full_dataset.classes}")

# 70% train, 15% val, 15% test
total = len(full_dataset)
train_size = int(0.7 * total)
val_size = int(0.15 * total)
test_size = total - train_size - val_size

train_dataset, val_dataset, test_dataset = random_split(
    full_dataset,
    [train_size, val_size, test_size],
    generator=torch.Generator().manual_seed(42)
)

# Apply training augmentation to train set
class AugmentedSubset(torch.utils.data.Dataset):
    """Wraps a Subset and applies a different transform."""
    def __init__(self, subset, transform):
        self.subset = subset
        self.transform = transform

    def __len__(self):
        return len(self.subset)

    def __getitem__(self, idx):
        # Get the original image (before any transform)
        img_path, label = self.subset.dataset.samples[self.subset.indices[idx]]
        img = Image.open(img_path).convert("RGB")
        img = self.transform(img)
        return img, label

train_dataset_aug = AugmentedSubset(train_dataset, training_transform)

train_loader = DataLoader(train_dataset_aug, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

print(f"Train: {len(train_dataset_aug)}, Val: {len(val_dataset)}, Test: {len(test_dataset)}")

In [0]:
# ============================================================
# 5. Build ResNet18
# ============================================================
model = models.resnet18(pretrained=True)

# Freeze early layers (optional, speeds up training)
for param in model.parameters():
    param.requires_grad = False

# Replace final layer for binary classification
model.fc = nn.Linear(512, NUM_CLASSES)

# Unfreeze fc layer
for param in model.fc.parameters():
    param.requires_grad = True

model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.fc.parameters(), lr=LR)

# ============================================================
# 6. Train
# ============================================================
def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    return running_loss / total, correct / total


def evaluate(model, loader, criterion):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

    return running_loss / total, correct / total


print("\n" + "="*60)
print("TRAINING ResNet18 on Cats vs Dogs")
print("="*60)

best_val_acc = 0.0
train_losses, val_losses = [], []
train_accs, val_accs = [], []

for epoch in range(EPOCHS):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer)
    val_loss, val_acc = evaluate(model, val_loader, criterion)

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accs.append(train_acc)
    val_accs.append(val_acc)

    print(f"Epoch [{epoch+1}/{EPOCHS}] "
          f"Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | "
          f"Val Loss: {val_loss:.4f} Acc: {val_acc:.4f}")

    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "resnet18_cats_dogs_best.pth")
        print(f"  ✓ Saved best model (val_acc={val_acc:.4f})")


In [0]:
# ============================================================
# 8. Test set evaluation
# ============================================================
model.load_state_dict(torch.load("resnet18_cats_dogs_best.pth", map_location=device))
model.eval()

test_loss, test_acc = evaluate(model, test_loader, criterion)
print(f"\n{'='*60}")
print(f"TEST ACCURACY: {test_acc:.4f}")
print(f"{'='*60}")

# Plot training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(train_losses, label='Train')
ax1.plot(val_losses, label='Val')
ax1.set_title('Loss')
ax1.legend()
ax2.plot(train_accs, label='Train')
ax2.plot(val_accs, label='Val')
ax2.set_title('Accuracy')
ax2.legend()
plt.tight_layout()
plt.savefig("training_curves.png", dpi=150)
plt.show()


In [0]:
# ---- Recreate the EXACT same architecture ----
model = models.resnet18(weights=None)  # Don't load pretrained weights
model.fc = nn.Linear(model.fc.in_features, 2)  # 2 classes: Cat, Dog

# ---- Load the saved weights ----
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model.load_state_dict(torch.load("resnet18_cats_dogs_best.pth", map_location=device))
model.to(device)
model.eval()  # Set to evaluation mode

print("✓ Model loaded successfully!")

In [0]:
# ============================================================
# 9. SHAP EXPLANATION (Using Captum)
# ============================================================
from captum.attr import GradientShap
from captum.attr import visualization as viz

print("\n" + "="*60)
print("COMPUTING SHAP VALUES")
print("="*60)


# Get test images as tensors
def get_samples_from_loader(loader, n_images=8):
    images_list = []
    labels_list = []
    collected = 0
    for images, labels in loader:
        for i in range(images.shape[0]):
            images_list.append(images[i])
            labels_list.append(labels[i].item())
            collected += 1
            if collected >= n_images:
                return images_list, labels_list
    return images_list, labels_list

sample_images, sample_labels = get_samples_from_loader(train_loader, n_images=1024)
print(f"Explaining {len(sample_images)} images...")

# Sanity check predictions
print("\nPredictions:")
for i in range(len(sample_images)):
    inp = sample_images[i].unsqueeze(0).to(device)
    with torch.no_grad():
        probs = torch.softmax(model(inp), dim=1)
    pred = probs.argmax(dim=1).item()
    conf = probs[0][pred].item()
    status = "✓" if pred == sample_labels[i] else "✗"
    print(f"  Image {i}: True={CLASS_NAMES[sample_labels[i]]}, "
          f"Pred={CLASS_NAMES[pred]}, "
          f"Conf={conf:.3f} {status}")

# ---- SHAP using Captum's GradientShap ----
gradient_shap = GradientShap(model)

all_attributions = []
all_predictions = []

for i in range(len(sample_images)):
    inp = sample_images[i].unsqueeze(0).to(device)
    inp.requires_grad = True

    with torch.no_grad():
        pred = model(inp).argmax(dim=1).item()
    all_predictions.append(pred)

    # Baselines: black and white images
    baselines = torch.cat([
        torch.zeros_like(inp),
        torch.ones_like(inp)
    ], dim=0).to(device)

    attr = gradient_shap.attribute(
        inp,
        baselines=baselines,
        target=pred,
        n_samples=50,
        stdevs=0.0001
    )

    all_attributions.append(attr.squeeze(0).cpu().detach().numpy())
    print(f"  ✓ Image {i} SHAP done - Pred: {CLASS_NAMES[pred]}")

print("\n✓ All SHAP values computed!")

In [0]:
shap_values = np.array(all_attributions)

Aggregate SHAP values to super-pixels

In [0]:

# Divide each image into a grid and average SHAP within each cell
def aggregate_to_grid(attributions, grid_size=4):
    """Reduce (3, 200, 200) → (grid_size, grid_size) per image."""
    all_grids = []
    h, w = 200, 200
    cell_h = h // grid_size
    cell_w = w // grid_size
    
    for attr in attributions:
        # Average across channels first: (3, 200, 200) → (200, 200)
        attr_mean = np.mean(attr, axis=0)
        
        grid = np.zeros((grid_size, grid_size))
        for r in range(grid_size):
            for c in range(grid_size):
                region = attr_mean[r*cell_h:(r+1)*cell_h, c*cell_w:(c+1)*cell_w]
                grid[r, c] = region.mean()
        
        all_grids.append(grid.flatten())
    
    return np.array(all_grids)
shap_values= aggregate_to_grid(all_attributions, grid_size=10)
print(f"Shape: {shap_values.shape}")  # (8, 16)

Deviance residuals

In [0]:
# y_true: actual labels (0=Cat, 1=Dog)
y_true = np.array(sample_labels)

# y_prob: predicted probability of class 1 (Dog)
y_prob = []
for i in range(len(sample_images)):
    inp = sample_images[i].unsqueeze(0).to(device)
    with torch.no_grad():
        probs = torch.softmax(model(inp), dim=1)
    y_prob.append(probs[0][1].item())  # Probability of Dog (class 1)
y_prob = np.array(y_prob)

# Clip to avoid log(0)
y_prob_clipped = np.clip(y_prob, 1e-7, 1 - 1e-7)

# Deviance residuals
resid = np.sign(y_true - y_prob_clipped) * np.sqrt(
    -2 * (y_true * np.log(y_prob_clipped) + (1 - y_true) * np.log(1 - y_prob_clipped))
)

### IOM

Test the normality of the SHAP values and the residuals.

In [0]:
from pingouin import multivariate_normality
from scipy import stats
import seaborn as sns

In [0]:
print("SHAP values:", multivariate_normality(shap_values).pval), 
print("Residuals:", stats.shapiro(resid.reshape(-1)).pvalue)

Define the normalizing flows function.

In [0]:
iom = InfluentialOutlierMetric(shap_values, resid,
                               2, 2, 2, 
                               4, 4, 2, 
                               lambdas=np.concatenate([[0], np.exp(np.linspace(-3, 10, 50))]),
                               lambdas_resid=np.concatenate([[0], np.exp(np.linspace(-3, 10, 50))]),
                               epoch=500, epoch_resid=500)

Tune \\(\lambda\\).

In [0]:
# Tune for lambda
iom.find_best_lambda(alpha=0.05)

In [0]:
iom.find_best_lambda_resid(alpha=0.05)

In [0]:
# Compute thresholds
thresholds = iom.find_threshold(alpha=[0.05, 0.01])

# Compute IOM
IOM_ = iom.IOM()

Plot the IOM vs. the training index.

In [0]:
sns.scatterplot(x=pd.Series(range(len(IOM_)), name='index'), y=IOM_)
plt.axhline(y=thresholds[0], color='green')
plt.axhline(y=thresholds[1], color='red')
for i, txt in enumerate(range(len(IOM_))):
    plt.annotate(txt, (i, IOM_[i]))

In [0]:
# ============================================================
# Choose which images to plot by index
# ============================================================
selected_indices = [212, 611, 928, 997, 70, 201, 456, 610, 784]  # <-- CHANGE these to any indices you want

fig, axes = plt.subplots(len(selected_indices), 3, figsize=(14, 4 * len(selected_indices)))

# Handle case where only 1 image selected (axes won't be 2D)
if len(selected_indices) == 1:
    axes = axes.reshape(1, -1)

for row, i in enumerate(selected_indices):
    img_np = sample_images[i].permute(1, 2, 0).numpy()
    img_np = np.clip(img_np, 0, 1)

    attr = all_attributions[i]
    attr_mean = np.mean(attr, axis=0)  # Average across channels

    pred = all_predictions[i]
    true = sample_labels[i]
    status = "✓" if pred == true else "✗"

    # --------------------------------------------------------
    # Create 10x10 super-pixel SHAP values
    # --------------------------------------------------------
    h, w = attr_mean.shape
    block_h = h // 10
    block_w = w // 10

    superpixel_shap = np.zeros((10, 10))
    for br in range(10):
        for bc in range(10):
            r_start = br * block_h
            r_end = r_start + block_h if br < 9 else h  # last block absorbs remainder
            c_start = bc * block_w
            c_end = c_start + block_w if bc < 9 else w
            superpixel_shap[br, bc] = np.mean(attr_mean[r_start:r_end, c_start:c_end])

    # Symmetric color scale based on super-pixel values
    vmax = max(abs(superpixel_shap.min()), abs(superpixel_shap.max()))
    if vmax == 0:
        vmax = 1

    # Column 1: Original
    axes[row, 0].imshow(img_np)
    axes[row, 0].set_title(f"Image {i} | True: {CLASS_NAMES[true]}", fontsize=11)
    axes[row, 0].axis('off')

    # Column 2: 10x10 Super-pixel SHAP heatmap
    im = axes[row, 1].imshow(superpixel_shap, cmap='seismic', vmin=-vmax, vmax=vmax,
                              interpolation='nearest')
    axes[row, 1].set_title(f"SHAP (10×10) → {CLASS_NAMES[pred]} {status}", fontsize=11)
    axes[row, 1].set_xticks(np.arange(10))
    axes[row, 1].set_yticks(np.arange(10))
    axes[row, 1].tick_params(labelsize=7)
    # Annotate each super-pixel cell with its value
    for br in range(10):
        for bc in range(10):
            val = superpixel_shap[br, bc]
            text_color = 'white' if abs(val) > vmax * 0.6 else 'black'
            axes[row, 1].text(bc, br, f"{val:.3f}", ha='center', va='center',
                              fontsize=5, color=text_color, fontweight='bold')
    plt.colorbar(im, ax=axes[row, 1], fraction=0.046, pad=0.04)

    # Column 3: Overlay (upscale super-pixel grid back to original resolution)
    superpixel_upscaled = np.repeat(np.repeat(superpixel_shap, block_h, axis=0),
                                     block_w, axis=1)
    # Pad if image size isn't perfectly divisible by 10
    if superpixel_upscaled.shape[0] < h or superpixel_upscaled.shape[1] < w:
        padded = np.zeros_like(attr_mean)
        padded[:superpixel_upscaled.shape[0], :superpixel_upscaled.shape[1]] = superpixel_upscaled
        # Fill remainder rows/cols with nearest super-pixel value
        padded[superpixel_upscaled.shape[0]:, :] = padded[superpixel_upscaled.shape[0]-1:superpixel_upscaled.shape[0], :]
        padded[:, superpixel_upscaled.shape[1]:] = padded[:, superpixel_upscaled.shape[1]-1:superpixel_upscaled.shape[1]]
        superpixel_upscaled = padded

    axes[row, 2].imshow(img_np)
    axes[row, 2].imshow(superpixel_upscaled, cmap='seismic', alpha=0.5,
                         vmin=-vmax, vmax=vmax)
    axes[row, 2].set_title(f"Overlay (10×10 blocks)", fontsize=11)
    axes[row, 2].axis('off')

plt.tight_layout()
plt.savefig("shap_selected_superpixel_10x10.png", dpi=150, bbox_inches='tight')
plt.show()

## Influence functions

In [0]:
from captum.influence import NaiveInfluenceFunction
from torch.utils.data import Subset

In [0]:
class GPUDataset(torch.utils.data.Dataset):
    def __init__(self, dataset, device):
        self.dataset = dataset
        self.device = device

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        data, target = self.dataset[idx]
        
        # Handle data
        if isinstance(data, torch.Tensor):
            data = data.to(self.device)
        
        # Handle target - convert int to tensor on GPU
        if isinstance(target, (int, float)):
            target = torch.tensor(target, device=self.device)
        elif isinstance(target, torch.Tensor):
            target = target.to(self.device)
        
        return data, target

subset_dataset = Subset(train_loader.dataset, list(range(1024)))
gpu_subset_dataset = GPUDataset(subset_dataset, device)

In [0]:
influence = NaiveInfluenceFunction(
    model=model,
    train_dataset=gpu_subset_dataset,
    checkpoint="resnet18_cats_dogs_best.pth",
    loss_fn=nn.CrossEntropyLoss(reduction="none"),
    batch_size=32,
    layers=["fc"],
)

In [0]:
subset_dataset_test = Subset(test_loader.dataset, list(range(1024)))
gpu_subset_dataset_test = GPUDataset(subset_dataset, device)

In [0]:
test_dl = DataLoader(gpu_subset_dataset_test, batch_size=64, shuffle=False)

all_scores = []

for i, batch in enumerate(test_dl):
    print(i)
    batch_data, batch_targets = batch
    scores = influence.influence(inputs=(batch_data, batch_targets))
    all_scores.append(scores.cpu())

influence_scores = torch.cat(all_scores, dim=0)

In [0]:
avg_influence = influence_scores.mean(dim=0)

In [0]:
[212, 611, 928, 997, 70, 201, 456, 610, 784]

In [0]:
sns.scatterplot(x=pd.Series(range(len(IOM_)), name='index'), y=IOM_)
plt.axhline(y=thresholds[0], color='green')
plt.axhline(y=thresholds[1], color='red')
for i, txt in enumerate(range(len(IOM_))):
    plt.annotate(txt, (i, IOM_[i]))

In [0]:
sns.scatterplot(x=pd.Series(range(len(avg_influence)), name='index'), y=avg_influence)
plt.ylabel('Influence')
for i, txt in enumerate(range(len(avg_influence))):
    plt.annotate(txt, (i, avg_influence[i]))

In [0]:
selected_indices = [818, 1007, 544]

fig, axes = plt.subplots(len(selected_indices), 1, figsize=(5, 4 * len(selected_indices)))

# Handle case where only 1 image selected
if len(selected_indices) == 1:
    axes = [axes]

for row, i in enumerate(selected_indices):
    img_np = sample_images[i].permute(1, 2, 0).numpy()
    img_np = np.clip(img_np, 0, 1)

    true = sample_labels[i]
    pred = all_predictions[i]
    status = "✓" if pred == true else "✗"

    axes[row].imshow(img_np)
    axes[row].set_title(
        f"Image {i} | True: {CLASS_NAMES[true]} | Pred: {CLASS_NAMES[pred]} {status}", 
        fontsize=11
    )
    axes[row].axis('off')

plt.tight_layout()
plt.show()